<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 5 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">批量导入、失败与重试</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">使用统一订单样本，观察 SQL、结果与验收证据。请按顺序运行单元。</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">目标 Doris 4.1.3 · 订单数据 · 独立实验库</span>
</div>

完成后，你将导入 10 张 WWI 历史表（701,846 行），再使用 Stream Load 导入十笔模拟新订单，核对同 label 重试不重复追加，并观察错误批次被拒绝。请按顺序运行；本实验不包含 Kafka 或 CDC 环境。

[讲义](course5_batch_and_streaming_ingestion.md) · [课程入口](../README.md)


## 实验范围

仅重建 orders_imported 和 wwi_orders、wwi_order_lines、wwi_customers、wwi_products、wwi_invoices、wwi_invoice_lines、wwi_customer_transactions、wwi_payment_methods、wwi_transaction_types、wwi_delivery_methods。历史 Parquet 需按 datasets/README.md 本地准备；模拟 CSV 随仓库提供。不依赖 S3 凭据。课程工具会自动配置沙箱的 BE HTTP 接入地址。本 Lab 不运行 Kafka、CDC、对象存储或 Group Commit；这些路径的概念与选择见讲义。


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows
from dw_course.ui import show_sql, show_response

lab = connect_sandbox()


from uuid import uuid4



## 1. 从历史业务表开始

先校验本地 10 个文件的 SHA-256，通过后才重建本节的历史表。不会恢复 SQL Server，也不访问课程桶。
文件来源为微软 WWI，日期不改写；客户交易的收款是账户级记录，不自动分摊到订单。
阅读每张表的建表语句，分清 Orders 与 OrderLines、Invoices 与 InvoiceLines 的粒度。


In [ ]:
from dw_course.wwi import manifest, parquet_paths, parquet_ddl
paths = parquet_paths()  # 缺文件或校验和不符会在删表之前停止
for name, metadata in manifest()["tables"].items():
    target = "wwi_" + name
    lab.execute("DROP TABLE IF EXISTS " + target)
    ddl = parquet_ddl(name, target)
    show_sql("WWI 历史表：" + name, ddl)
    lab.execute(ddl)
    response = lab.stream_load(target, paths[name], "wwi_" + uuid4().hex, format="parquet")
    show_response(response)
    expect(response["Status"], "Success")
    expect(response["NumberLoadedRows"], metadata["rows"])
    expect(response["NumberFilteredRows"], 0)
    primary = metadata["primary"]
    expect(lab.query(f"SELECT COUNT(*), COUNT(DISTINCT {primary}) FROM {target}"),
           [(metadata["rows"], metadata["rows"])])


### 导入成功后，回答两个业务问题

1. 订单明细关联订单、客户和商品后，行数是否扩大或丢失？
2. 发票与收款是否表示同一种金额？下面按交易类型展示，保留原始记账正负方向。


In [ ]:
expect(lab.query("""
SELECT COUNT(*), SUM(l.Quantity*l.UnitPrice)
FROM wwi_order_lines l
JOIN wwi_orders o ON l.OrderID=o.OrderID
JOIN wwi_customers c ON o.CustomerID=c.CustomerID
JOIN wwi_products p ON l.StockItemID=p.StockItemID
"""), [(231412, "177634276.40")])
lab.sql("""
SELECT t.TransactionTypeName, COUNT(*) AS rows_count,
       SUM(c.TransactionAmount) AS ledger_amount,
       SUM(CASE WHEN c.InvoiceID IS NULL THEN 1 ELSE 0 END) AS no_invoice_link
FROM wwi_customer_transactions c
JOIN wwi_transaction_types t ON c.TransactionTypeID=t.TransactionTypeID
GROUP BY t.TransactionTypeName ORDER BY t.TransactionTypeName
""", title="发票记账与账户收款")
expect(lab.query("SELECT SUM(TransactionAmount), SUM(OutstandingBalance) FROM wwi_customer_transactions"),
       [("267011.44", "267011.44")])
expect(lab.query("SELECT COUNT(*) FROM wwi_invoices WHERE ConfirmedDeliveryTime IS NOT NULL"), [(70426,)])
lab.sql("""
SELECT o.OrderDate, COUNT(DISTINCT o.OrderID) AS orders,
       SUM(l.Quantity*l.UnitPrice) AS order_amount
FROM wwi_orders o JOIN wwi_order_lines l ON o.OrderID=l.OrderID
GROUP BY o.OrderDate ORDER BY o.OrderDate LIMIT 10
""", title="完整历史上的每日订单分析")


### 从历史存量过渡到新订单

接下来的 CSV 是课程生成的 900001–900010，不是从 WWI 推断出的“未支付订单”。
它引用 WWI 客户 1–10、商品 1–10；模拟价格、地区和事件时间单独定义。
data_source=COURSE_SIMULATION；历史保持在 wwi_*，两种来源不混算。
预期十笔新订单金额 1400.00、初始支付为零。


## 2. 导入模拟新订单

显式 off_mode。先保留原始响应，再同时核对业务数据，不能只判断 HTTP 200。


In [ ]:
lab.execute("DROP TABLE IF EXISTS orders_imported")
ddl = order_ddl("orders_imported")
show_sql("建表 SQL", ddl)
lab.execute(ddl)
label = "dw_l1_" + uuid4().hex
columns = ",".join(ORDER_COLUMNS)
result = lab.stream_load("orders_imported", COURSE_ROOT / "datasets/orders.csv", label, columns)
show_response(result)
expect(result["Status"], "Success")
expect(result["NumberLoadedRows"], 10)
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_imported"), [(10,"1400.00")])


## 3. 同一批次用同 label 重试

在 label 仍有效的时间内立即重试。导入批次身份不等于永久业务去重，Unique Key 和事件版本在 D06 另行讨论。


In [ ]:
retry = lab.stream_load("orders_imported", COURSE_ROOT / "datasets/orders.csv", label, columns)
show_response(retry)
expect(retry["Status"], "Label Already Exists")
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_imported"), [(10,"1400.00")])


## 4. 观察坏数据整批拒绝

新 label 导入两行，其中一行金额无法转换。strict_mode=true 且 max_filter_ratio=0；预期整批失败，原来的十行不变。


In [ ]:
rejected = lab.stream_load("orders_imported", COURSE_ROOT / "datasets/malformed_orders.csv",
                           "dw_bad_" + uuid4().hex, columns)
show_response(rejected)
expect(rejected["Status"], "Fail")
expect(rejected["NumberFilteredRows"], 1)
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_imported"), [(10,"1400.00")])
lab.close()


## 完成与排查

记录 ErrorURL 并在可信实验环境及时查看；不要将临时错误日志当永久拒收表。若出现 Publish Timeout 或不同状态，应保留响应核对事务与可见性，不能删除后重导掩盖问题。D09-A 将原始输入保留后再分流。


## 自己动手与验收

查询 WWI 商品销售额排名，解释明细行数为什么不是订单数。
完整历史金额、账户收款和模拟订单金额分别属于哪张表？为什么不能把收款负数直接解释为退款？
完成历史 10 表导入、关联校验、模拟 CSV 的同 label 重试与坏数据拒绝，才算完成本 Lab。
下一步 D09-A 验证模拟订单的客户引用和质量，D06 接入支付、退款及配送事件。
